# NYC TLC Data Exploration (Track B)


In [8]:
import pandas as pd
import numpy as np

# Set pandas display options to show all columns
pd.set_option('display.max_columns', None)

# Load the yellow taxi data
yellow_file = 'data/yellow_tripdata_2026-04.parquet'
df_yellow = pd.read_parquet(yellow_file)

print(f"Total Rows: {len(df_yellow):,}")
print(f"Total Columns: {len(df_yellow.columns)}")

# Preview the first 5 rows
df_yellow.head()

Total Rows: 3,831,240
Total Columns: 20


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2026-04-01 00:40:05,2026-04-01 00:52:44,1.0,2.80,1.0,N,237,68,1,15.6,4.25,0.5,4.25,0.00,1.0,25.60,2.5,0.0,0.75
1,2,2026-04-01 00:09:19,2026-04-01 00:21:29,1.0,7.37,1.0,N,138,75,1,28.2,6.00,0.5,9.03,7.46,1.0,54.19,0.0,2.0,0.00
2,2,2026-04-01 00:15:29,2026-04-01 00:34:14,1.0,7.66,1.0,N,138,112,1,31.7,6.00,0.5,5.00,0.00,1.0,46.20,0.0,2.0,0.00
3,1,2026-04-01 00:14:20,2026-04-01 00:27:49,0.0,7.90,1.0,N,138,262,1,31.0,10.50,0.5,10.09,7.46,1.0,60.55,2.5,2.0,0.00
4,2,2026-04-01 00:04:53,2026-04-01 00:11:54,1.0,1.34,1.0,N,230,234,1,8.6,1.00,0.5,2.87,0.00,1.0,17.22,2.5,0.0,0.75


## 1. Data Profiling 
Before building metrics, you need to validate the data. Let's check for missing values and look at basic statistics for numerical columns to identify any anomalies (like negative fares or zero distances).

In [9]:
# Check for missing (null) values in each column
print("Missing Values:")
df_yellow.isnull().sum()

Missing Values:


VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          799786
trip_distance                 0
RatecodeID               799786
store_and_fwd_flag       799786
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     799786
Airport_fee              799786
cbd_congestion_fee            0
dtype: int64

In [10]:
# Look at basic statistics for numerical columns
# Notice anything strange? (e.g., negative fare amounts or 0 passenger counts)
df_yellow[['passenger_count', 'trip_distance', 'fare_amount', 'total_amount']].describe()

,passenger_count,trip_distance,fare_amount,total_amount
count,3.031454e+06,3.831240e+06,3.831240e+06,3.831240e+06
mean,1.245977e+00,5.179852e+00,2.108398e+01,3.000036e+01
std,6.495457e-01,4.927210e+02,1.830420e+01,2.211713e+01
min,0.000000e+00,0.000000e+00,-1.274900e+03,-1.278400e+03
25%,1.000000e+00,1.030000e+00,1.000000e+01,1.735000e+01
50%,1.000000e+00,1.850000e+00,1.560000e+01,2.358000e+01
75%,1.000000e+00,3.790000e+00,2.610000e+01,3.446000e+01
max,9.000000e+00,2.815761e+05,1.442200e+03,1.452660e+03


## 2. Deriving Operational Metrics 
To define useful operational metrics, we often need to calculate new fields. For example, let's calculate the **trip duration** in minutes.

In [11]:
# Calculate trip duration in minutes
df_yellow['trip_duration_minutes'] = (df_yellow['tpep_dropoff_datetime'] - df_yellow['tpep_pickup_datetime']).dt.total_seconds() / 60

# Look at the statistics of our new metric
df_yellow['trip_duration_minutes'].describe()

count    3.831240e+06
mean     1.801072e+01
std      2.477928e+01
min      0.000000e+00
25%      8.316667e+00
50%      1.398333e+01
75%      2.238333e+01
max      9.983933e+03
Name: trip_duration_minutes, dtype: float64

## Next Steps for the Pipeline
1. **Clean the Data**: Create a new DataFrame (`df_clean`) that filters out the bad data you found above (e.g., `df_clean = df_yellow[df_yellow['fare_amount'] > 0]`).
2. **Calculate KPIs**: Group by `PULocationID` (Pickup Location) or date to find average trip durations, total revenue, or total trips.
3. **Refine**: Check out the other parquet files (like `green_tripdata`) and see if you want to join them or compare them!

In [12]:
# 1. Calculate the speed (MPH)
# Distance is in miles, duration is in minutes (divide by 60 for hours)
df_yellow['speed_mph'] = df_yellow['trip_distance'] / (df_yellow['trip_duration_minutes'] / 60)

# 2. Load the new CSV we downloaded (Class 5 Check!)
df_zones = pd.read_csv('data/taxi_zone_lookup.csv')

# 3. Join the zones to our yellow taxi data to get real neighborhood names (Class 7 Check!)
df_joined = df_yellow.merge(df_zones, left_on='PULocationID', right_on='LocationID', how='left')
df_joined = df_joined.rename(columns={'Zone': 'Pickup_Neighborhood', 'Borough': 'Pickup_Borough'})

# Display the routes with their speeds
df_joined[['Pickup_Neighborhood', 'trip_distance', 'trip_duration_minutes', 'speed_mph']].head()


,Pickup_Neighborhood,trip_distance,trip_duration_minutes,speed_mph
0,Upper East Side South,2.80,12.650000,13.280632
1,LaGuardia Airport,7.37,12.166667,36.345205
2,LaGuardia Airport,7.66,18.750000,24.512000
3,LaGuardia Airport,7.90,13.483333,35.154512
4,Times Sq/Theatre District,1.34,7.016667,11.458432
